### **Notebook Goal** 

**Purpose**: Transform the raw Freddie Mac 2018 mortgage dataset *02_freddie_mac_data_preparation.ipynb* into two model-ready analytical datasets, while keeping the heavy longitudinal processing centralized in one preparation pipeline.

This notebook acts as our automated data-preparation pipeline: 
$$ \boxed {Raw Data → Business Rules → Target Construction → Analytical Datasets → Saved Processed Data.} $$

### **Data inspection**

### Locate and inspect the two files

In [1]:
# FREDDIE MAC 2018 — RAW DATA INSPECTION
# Purpose:
# Locate the origination and monthly performance datasets without loading the full performance dataset into memory.

from pathlib import Path
import os

# Current notebook working directory.
print("Current working directory:")
print(Path.cwd())

# Project root.
# notebooks/ -> project root
PROJECT_ROOT = Path.cwd().parent

# Freddie Mac 2018 raw-data directory.
FREDDIE_DIR = PROJECT_ROOT / "data" / "raw" / "freddie_mac" / "2018"

ORIG_PATH = FREDDIE_DIR / "sample_orig_2018.txt"
PERF_PATH = FREDDIE_DIR / "sample_perf_2018.txt" 

# Verify that both files exist.
print("\nOrigination file exists:", ORIG_PATH.exists())
print("Performance file exists:", PERF_PATH.exists())

# Check file sizes without opening the datasets.
print("\nFile sizes:")
print(f"Origination: {ORIG_PATH.stat().st_size / 1024**2:.2f} MB")
print(f"Performance: {PERF_PATH.stat().st_size / 1024**2:.2f} MB")

Current working directory:
c:\Users\guima\OneDrive\Documents\AI_project\banking_services_ai_assistant\notebooks

Origination file exists: True
Performance file exists: True

File sizes:
Origination: 6.05 MB
Performance: 215.45 MB


### Inspect raw records without loading the datasets

In [2]:
# INSPECT RAW FILE STRUCTURE
# Read only the first few lines from each source file.

print("ORIGINATION FILE — FIRST 3 RECORDS\n")

with open(ORIG_PATH, "r", encoding="utf-8") as file:
    for _ in range(3):
        print(file.readline().strip())


print("\n" + "=" * 100)
print("PERFORMANCE FILE — FIRST 3 RECORDS\n")

with open(PERF_PATH, "r", encoding="utf-8") as file:
    for _ in range(3):
        print(file.readline().strip())

ORIGINATION FILE — FIRST 3 RECORDS

654|201803|N|204802||0|1|P|77|35|50000|77|4.5|R|N|FRM|KY|SF|421|F18Q10000028|P|360|1|OTHER|N|||N|7|N|9999
693|201803|N|203302|24340|0|1|P|80|41|132000|80|3.25|R|N|FRM|MI|SF|493|F18Q10000052|P|180|2|OTHER|N|||N|1|N|9999
757|201803|Y|204802||25|1|P|97|10|28000|97|4|R|N|FRM|IA|SF|513|F18Q10000084|P|360|1|OTHER|N||H|N|7|N|9999

PERFORMANCE FILE — FIRST 3 RECORDS

F18Q10000028|201802|50000.00|00|0|360|||||4.500|0.00||||||||||||||76||||||50000.00|7|OTHER|
F18Q10000028|201803|50000.00|00|1|359|||||4.500|0.00||||||||||||||76||||||50000.00|7|OTHER|
F18Q10000028|201804|50000.00|00|2|358|||||4.500|0.00||||||||||||||73||||||50000.00|7|OTHER|


### Define initial freddie_mac dataset columns

In [3]:
# DEFINE OFFICIAL FREDDIE MAC COLUMN NAMES
# These column names follow the Pre-July 2026 Freddie Mac Origination and Monthly Performance file layouts.

ORIG_COLUMNS = [
    "credit_score",
    "first_payment_date",
    "first_time_homebuyer_flag",
    "maturity_date",
    "msa",
    "mi_percentage",
    "number_of_units",
    "occupancy_status",
    "original_cltv",
    "original_dti",
    "original_upb",
    "original_ltv",
    "original_interest_rate",
    "channel",
    "ppm_flag",
    "amortization_type",
    "property_state",
    "property_type",
    "postal_code",
    "loan_sequence_number",
    "loan_purpose",
    "original_loan_term",
    "number_of_borrowers",
    "seller_name",
    "servicer_name",
    "super_conforming_flag",
    "pre_harp_loan_sequence_number",
    "program_indicator",
    "harp_indicator",
    "property_valuation_method",
    "interest_only_indicator",
    "mi_cancellation_indicator"
]


PERF_COLUMNS = [
    "loan_sequence_number",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_maturity",
    "defect_settlement_date",
    "modification_flag",
    "zero_balance_code",
    "zero_balance_effective_date",
    "current_interest_rate",
    "current_deferred_upb",
    "ddlpi",
    "mi_recoveries",
    "net_sales_proceeds",
    "non_mi_recoveries",
    "expenses",
    "legal_costs",
    "maintenance_preservation_costs",
    "taxes_insurance",
    "miscellaneous_expenses",
    "actual_loss",
    "modification_cost",
    "step_modification_flag",
    "deferred_payment_plan",
    "estimated_ltv",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "delinquency_due_to_disaster",
    "borrower_assistance_status",
    "current_month_modification_cost",
    "interest_bearing_upb",
    "mi_cancellation_indicator",        
    "servicer_name",                    
    "bankruptcy_cramdown_costs" 
]

print("Origination columns:", len(ORIG_COLUMNS))
print("Performance columns:", len(PERF_COLUMNS))

Origination columns: 32
Performance columns: 35


### Business definition of each variable

##### **Origination Data — Business Definitions**

| Column                          | Business definition                                                                         |
| ------------------------------- | ------------------------------------------------------------------------------------------- |
| `credit_score`                  | Borrower creditworthiness score at loan origination.                                        |
| `first_payment_date`            | Month when the borrower’s first mortgage payment is due.                                    |
| `first_time_homebuyer_flag`     | Indicates whether the borrower is purchasing a home for the first time.                     |
| `maturity_date`                 | Contractual month when the mortgage is scheduled to be fully repaid.                        |
| `msa`                           | Geographic market area where the property is located.                                       |
| `mi_percentage`                 | Percentage of the mortgage covered by mortgage insurance.                                   |
| `number_of_units`               | Number of residential units in the financed property.                                       |
| `occupancy_status`              | Indicates whether the property is owner-occupied, a second home, or an investment property. |
| `original_cltv`                 | Total mortgage debt relative to the property value at origination, including other liens.   |
| `original_dti`                  | Borrower debt obligations relative to income at origination.                                |
| `original_upb`                  | Original unpaid principal balance; the initial mortgage amount financed.                    |
| `original_ltv`                  | Original mortgage balance relative to the property value.                                   |
| `original_interest_rate`        | Interest rate charged when the mortgage was originated.                                     |
| `channel`                       | Business channel through which the mortgage was originated or acquired.                     |
| `ppm_flag`                      | Indicates whether the mortgage includes a prepayment penalty.                               |
| `amortization_type`             | Defines the mortgage repayment structure, such as fixed-rate amortization.                  |
| `property_state`                | U.S. state where the financed property is located.                                          |
| `property_type`                 | Type of property securing the mortgage.                                                     |
| `postal_code`                   | Geographic postal-code identifier of the property.                                          |
| `loan_sequence_number`          | Unique mortgage identifier used to link origination and monthly performance records.        |
| `loan_purpose`                  | Purpose of the mortgage, such as purchase, refinance, or cash-out refinance.                |
| `original_loan_term`            | Original contractual loan duration in months.                                               |
| `number_of_borrowers`           | Number of borrowers legally responsible for the mortgage.                                   |
| `seller_name`                   | Institution that sold the mortgage to Freddie Mac.                                          |
| `servicer_name`                 | Institution responsible for servicing the mortgage.                                         |
| `super_conforming_flag`         | Indicates whether the mortgage falls within the super-conforming loan category.             |
| `pre_harp_loan_sequence_number` | Identifier of the previous loan associated with a HARP refinance.                           |
| `program_indicator`             | Identifies whether the mortgage belongs to a specific Freddie Mac program.                  |
| `harp_indicator`                | Indicates whether the loan was originated through HARP.                                     |
| `property_valuation_method`     | Method used to determine the property's value during underwriting.                          |
| `interest_only_indicator`       | Indicates whether the mortgage contains an interest-only payment feature.                   |
| `mi_cancellation_indicator`     | Indicates whether mortgage insurance cancellation applies to the loan.                      |


#### **Monthly Performance Data — Business Definitions**

| Column                            | Business definition                                                                            |
| --------------------------------- | ---------------------------------------------------------------------------------------------- |
| `loan_sequence_number`            | Unique mortgage identifier linking monthly performance to origination data.                    |
| `monthly_reporting_period`        | Month corresponding to the reported loan performance.                                          |
| `current_actual_upb`              | Outstanding principal balance remaining on the mortgage for the reporting month.               |
| `current_loan_delinquency_status` | Current payment delinquency status of the borrower.                                            |
| `loan_age`                        | Number of months elapsed since the mortgage began.                                             |
| `remaining_months_to_maturity`    | Number of contractual months remaining until maturity.                                         |
| `defect_settlement_date`          | Date associated with settlement of an identified loan defect.                                  |
| `modification_flag`               | Indicates whether the mortgage has undergone a loan modification.                              |
| `zero_balance_code`               | Reason the mortgage balance reached zero, such as payoff or disposition.                       |
| `zero_balance_effective_date`     | Date when the mortgage balance became zero.                                                    |
| `current_interest_rate`           | Interest rate applicable during the reporting month.                                           |
| `current_deferred_upb`            | Portion of principal whose repayment has been deferred.                                        |
| `ddlpi`                           | Due date of the last installment successfully paid by the borrower.                            |
| `mi_recoveries`                   | Amount recovered through mortgage insurance after a credit event.                              |
| `net_sales_proceeds`              | Net proceeds obtained from the sale or disposition of the property.                            |
| `non_mi_recoveries`               | Recoveries received from sources other than mortgage insurance.                                |
| `expenses`                        | Expenses associated with resolving or disposing of a distressed mortgage.                      |
| `legal_costs`                     | Legal expenses incurred during loan resolution or foreclosure.                                 |
| `maintenance_preservation_costs`  | Costs incurred to maintain or preserve the property.                                           |
| `taxes_insurance`                 | Taxes and insurance expenses associated with the property or loan resolution.                  |
| `miscellaneous_expenses`          | Other expenses incurred during the resolution process.                                         |
| `actual_loss`                     | Realized financial loss calculated after recoveries and expenses.                              |
| `modification_cost`               | Cost associated with modifying the mortgage terms.                                             |
| `step_modification_flag`          | Indicates whether the loan received a step-rate modification.                                  |
| `deferred_payment_plan`           | Indicates whether the borrower received a deferred-payment arrangement.                        |
| `estimated_ltv`                   | Estimated current loan balance relative to the property's estimated value.                     |
| `zero_balance_removal_upb`        | Principal balance remaining immediately before the mortgage was removed from active reporting. |
| `delinquent_accrued_interest`     | Interest accumulated while the mortgage was delinquent.                                        |
| `delinquency_due_to_disaster`     | Indicates whether delinquency is associated with a disaster event.                             |
| `borrower_assistance_status`      | Indicates the borrower's current loss-mitigation or assistance status.                         |
| `current_month_modification_cost` | Modification-related cost recognized during the reporting month.                               |
| `interest_bearing_upb`            | Portion of the outstanding principal balance that continues to accrue interest.                |
| `mi_cancellation_indicator`       | Indicates whether mortgage insurance coverage has been cancelled for the loan |
| `servicer_name`              | Identifies the institution responsible for administering and servicing the mortgage |
| `bankruptcy_cramdown_costs` | Costs associated with a court-ordered reduction or modification of mortgage|



### Load origination & controlled performance sample

In [4]:
# LOAD DATA FOR INITIAL INSPECTION

import pandas as pd

# Import ORIGINATION dataset

orig_df = pd.read_csv(
    ORIG_PATH,
    sep="|",
    header=None,
    names=ORIG_COLUMNS,
    low_memory=False
)

# Import PERFORMANCE dataset while loading only 10,000 rows for structural inspection

perf_sample = pd.read_csv(
    PERF_PATH,
    sep="|",
    header=None,
    names=PERF_COLUMNS,
    nrows=10_000,
    low_memory=False
)

print("Origination shape:", orig_df.shape)
print("Performance inspection sample:", perf_sample.shape)

Origination shape: (50000, 32)
Performance inspection sample: (10000, 35)


### Inspect the mapped data

In [5]:
# INSPECT MAPPED DATASETS

print("ORIGINATION DATA")
display(orig_df.head())

print("\nOrigination data types:")
display(orig_df.dtypes.to_frame("dtype"))


print("\nPERFORMANCE SAMPLE")
display(perf_sample.head())

print("\nPerformance data types:")
display(perf_sample.dtypes.to_frame("dtype"))

ORIGINATION DATA


,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mi_percentage,number_of_units,occupancy_status,original_cltv,original_dti,...,number_of_borrowers,seller_name,servicer_name,super_conforming_flag,pre_harp_loan_sequence_number,program_indicator,harp_indicator,property_valuation_method,interest_only_indicator,mi_cancellation_indicator
0,654,201803,N,204802,NaN,0,1,P,77,35,...,1,OTHER,N,NaN,NaN,N,7,N,9999,NaN
1,693,201803,N,203302,24340.0,0,1,P,80,41,...,2,OTHER,N,NaN,NaN,N,1,N,9999,NaN
2,757,201803,Y,204802,NaN,25,1,P,97,10,...,1,OTHER,N,NaN,H,N,7,N,9999,NaN
3,807,201803,N,203302,NaN,0,1,P,26,38,...,2,OTHER,N,NaN,NaN,N,7,N,9999,NaN
4,812,201803,N,203302,19340.0,0,1,P,14,14,...,2,OTHER,N,NaN,NaN,N,1,N,9999,NaN



Origination data types:


,dtype
credit_score,int64
first_payment_date,int64
first_time_homebuyer_flag,str
maturity_date,int64
msa,float64
mi_percentage,int64
number_of_units,int64
occupancy_status,str
original_cltv,int64
original_dti,int64



PERFORMANCE SAMPLE


,loan_sequence_number,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_maturity,defect_settlement_date,modification_flag,zero_balance_code,zero_balance_effective_date,...,estimated_ltv,zero_balance_removal_upb,delinquent_accrued_interest,delinquency_due_to_disaster,borrower_assistance_status,current_month_modification_cost,interest_bearing_upb,mi_cancellation_indicator,servicer_name,bankruptcy_cramdown_costs
0,F18Q10000028,201802,50000.0,0,0,360,NaN,NaN,NaN,NaN,...,76,NaN,NaN,NaN,NaN,NaN,50000.0,7,OTHER,NaN
1,F18Q10000028,201803,50000.0,0,1,359,NaN,NaN,NaN,NaN,...,76,NaN,NaN,NaN,NaN,NaN,50000.0,7,OTHER,NaN
2,F18Q10000028,201804,50000.0,0,2,358,NaN,NaN,NaN,NaN,...,73,NaN,NaN,NaN,NaN,NaN,50000.0,7,OTHER,NaN
3,F18Q10000028,201805,50000.0,0,3,357,NaN,NaN,NaN,NaN,...,48,NaN,NaN,NaN,NaN,NaN,50000.0,7,OTHER,NaN
4,F18Q10000028,201806,50000.0,0,4,356,NaN,NaN,NaN,NaN,...,47,NaN,NaN,NaN,NaN,NaN,50000.0,7,OTHER,NaN



Performance data types:


,dtype
loan_sequence_number,str
monthly_reporting_period,int64
current_actual_upb,float64
current_loan_delinquency_status,int64
loan_age,int64
remaining_months_to_maturity,int64
defect_settlement_date,float64
modification_flag,str
zero_balance_code,float64
zero_balance_effective_date,float64


### Initial missingness inspection

In [6]:
# INITIAL MISSING-VALUE INSPECTION
orig_missing = (
    orig_df.isna()
    .sum()
    .to_frame("missing_count")
)

orig_missing["missing_pct"] = (
    orig_missing["missing_count"] / len(orig_df) * 100
).round(2)


perf_missing = (
    perf_sample.isna()
    .sum()
    .to_frame("missing_count")
)

perf_missing["missing_pct"] = (
    perf_missing["missing_count"] / len(perf_sample) * 100
).round(2)


print("ORIGINATION MISSING VALUES")
display(orig_missing.sort_values("missing_pct", ascending=False))


print("\nPERFORMANCE SAMPLE MISSING VALUES")
display(perf_missing.sort_values("missing_pct", ascending=False))

ORIGINATION MISSING VALUES


,missing_count,missing_pct
mi_cancellation_indicator,50000,100.00
super_conforming_flag,49489,98.98
pre_harp_loan_sequence_number,44331,88.66
msa,5170,10.34
maturity_date,0,0.00
credit_score,0,0.00
first_time_homebuyer_flag,0,0.00
first_payment_date,0,0.00
original_cltv,0,0.00
original_dti,0,0.00



PERFORMANCE SAMPLE MISSING VALUES


,missing_count,missing_pct
maintenance_preservation_costs,10000,100.00
expenses,10000,100.00
defect_settlement_date,10000,100.00
delinquent_accrued_interest,10000,100.00
actual_loss,10000,100.00
miscellaneous_expenses,10000,100.00
legal_costs,10000,100.00
non_mi_recoveries,10000,100.00
net_sales_proceeds,10000,100.00
mi_recoveries,10000,100.00


#### **Business target definition**

1. ***Mortgage PD Classification Model***

Business problem

Which currently performing mortgages are likely to default within a future period?

Mortgage PD Target — default_12m
Binary indicator representing whether a mortgage reaches serious delinquency (90+ days past due / 3+ monthly payments delinquent) within the following 12 months. The model estimates the probability of this event occurring.

Please note that we should eventually exclude mortgages that are already seriously delinquent at month 12. Otherwise, we'd be asking the model to predict future default for a loan that has already reached our definition of default.

So our eligible PD population should conceptually be: Not defaulted at observation month 12→Predict default during months 13–24.

2. ***EAD Regression Model***

The second model answers a different question:

If the mortgage defaults, what will the bank's outstanding exposure be at that point?

Mortgage EAD Target — ead_at_default
Continuous monetary value representing the outstanding unpaid principal balance when the mortgage first reaches the defined default event. The regression model estimates the bank's exposure at the time of default.

In [7]:
# ASSESS AVAILABILITY OF SERIOUS-DELINQUENCY EVENTS
# Business objective:
# Determine whether the 2018 Freddie Mac sample contains enough mortgages reaching 90+ days past due to support:

#   1. Mortgage PD classification
#   2. EAD regression

# Default definition for this project:
# A mortgage reaches default when delinquency status >= 3 (approximately 90+ days past due).

# IMPORTANT:
# We process the 215 MB performance file in chunks instead of loading the entire file into memory.

import pandas as pd

# Only these fields are required for this assessment.
event_columns = [
    "loan_sequence_number",
    "current_loan_delinquency_status",
    "current_actual_upb"
]

# Store loans that reach our default definition.
default_loans = set()

# Store the first observed EAD for each defaulted loan.
# This is only an availability check; we will construct the
# formal EAD target more carefully later.
default_ead = {}

# Track total monthly observations processed.
total_rows = 0

# Read manageable blocks from the performance file.
for chunk in pd.read_csv(
    PERF_PATH,
    sep="|",
    header=None,
    names=PERF_COLUMNS,
    usecols=event_columns,
    chunksize=100_000,
    low_memory=False
):

    total_rows += len(chunk)

    # Freddie Mac can contain non-numeric delinquency codes.
    # Invalid/non-numeric values become NaN rather than causing the processing step to fail.
    delinquency = pd.to_numeric(
        chunk["current_loan_delinquency_status"],
        errors="coerce"
    )

    # Identify observations at 90+ days past due.
    default_mask = delinquency >= 3

    default_events = chunk.loc[
        default_mask,
        ["loan_sequence_number", "current_actual_upb"]
    ]

    # Record each mortgage that ever reaches default.
    default_loans.update(
        default_events["loan_sequence_number"].dropna()
    )

    # Capture the first observed UPB at/after reaching default.
    # This gives us an initial indication of EAD availability.
    for loan_id, upb in default_events.itertuples(index=False):

        if loan_id not in default_ead:
            default_ead[loan_id] = upb


print(f"Monthly performance rows processed: {total_rows:,}")
print(f"Unique mortgages in origination data: {len(orig_df):,}")
print(f"Mortgages reaching 90+ DPD: {len(default_loans):,}")

Monthly performance rows processed: 2,059,564
Unique mortgages in origination data: 50,000
Mortgages reaching 90+ DPD: 2,708


In [8]:
# DEFAULT / EAD TARGET FEASIBILITY SUMMARY

total_loans = orig_df["loan_sequence_number"].nunique()

n_defaults = len(default_loans)

default_rate = (
    n_defaults / total_loans * 100
    if total_loans > 0
    else 0
)

# Convert the preliminary EAD dictionary into a DataFrame.
ead_check = pd.DataFrame(
    default_ead.items(),
    columns=["loan_sequence_number", "ead_at_default"]
)

# Make sure EAD is numeric.
ead_check["ead_at_default"] = pd.to_numeric(
    ead_check["ead_at_default"],
    errors="coerce"
)

valid_ead = ead_check["ead_at_default"].notna().sum()


print("MORTGAGE DEFAULT TARGET FEASIBILITY")
print(f"Total mortgages:          {total_loans:,}")
print(f"90+ DPD mortgages:        {n_defaults:,}")
print(f"Observed default rate:    {default_rate:.2f}%")

print("\n=== EAD TARGET FEASIBILITY ===")
print(f"Defaulted mortgages:      {n_defaults:,}")
print(f"Defaults with valid UPB:  {valid_ead:,}")

if n_defaults > 0:
    print(
        f"EAD coverage:             "
        f"{valid_ead / n_defaults * 100:.2f}%"
    )

# Inspect preliminary EAD distribution.
if valid_ead > 0:
    print("\nPreliminary EAD distribution:")
    display(ead_check["ead_at_default"].describe())

MORTGAGE DEFAULT TARGET FEASIBILITY
Total mortgages:          50,000
90+ DPD mortgages:        2,708
Observed default rate:    5.42%

=== EAD TARGET FEASIBILITY ===
Defaulted mortgages:      2,708
Defaults with valid UPB:  2,708
EAD coverage:             100.00%

Preliminary EAD distribution:


count      2708.000000
mean     222870.133061
std      126114.711424
min           0.000000
25%      127554.720000
50%      194971.565000
75%      299278.430000
max      707027.470000
Name: ead_at_default, dtype: float64

### Create loan-level performance table

In [9]:
# BUILD UNIQUE MONTH-12 OBSERVATION POINT

# Business objective:
# Create exactly one observation point per mortgage.

# Rule: Use the FIRST chronological record where loan_age = 12.

# This avoids duplicate analytical records caused by later loan modifications or term resets that can recreate loan_age 12.

pd_columns = [
    "loan_sequence_number",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_maturity",
    "current_interest_rate",
    "current_deferred_upb",
    "estimated_ltv",
    "interest_bearing_upb"
]

pd_chunks = []

for chunk in pd.read_csv(
    PERF_PATH,
    sep="|",
    header=None,
    names=PERF_COLUMNS,
    usecols=pd_columns,
    chunksize=100_000,
    low_memory=False
):

    # Convert fields required for chronological and risk analysis.
    numeric_columns = [
        "monthly_reporting_period",
        "current_actual_upb",
        "current_loan_delinquency_status",
        "loan_age",
        "remaining_months_to_maturity",
        "current_interest_rate",
        "current_deferred_upb",
        "estimated_ltv",
        "interest_bearing_upb"
    ]

    for column in numeric_columns:
        chunk[column] = pd.to_numeric(
            chunk[column],
            errors="coerce"
        )

    # We need the full available history because the target will be defined relative to the selected observation month.
    pd_chunks.append(chunk)


performance_history = pd.concat(
    pd_chunks,
    ignore_index=True
)

# Sort each mortgage chronologically.
performance_history = performance_history.sort_values(
    ["loan_sequence_number", "monthly_reporting_period"]
).reset_index(drop=True)


# Identify all age-12 observations.
month12_candidates = performance_history[
    performance_history["loan_age"] == 12
].copy()


# Keep only the FIRST chronological age-12 record for each loan.
observation_df = (
    month12_candidates
    .sort_values(
        ["loan_sequence_number", "monthly_reporting_period"]
    )
    .drop_duplicates(
        subset="loan_sequence_number",
        keep="first"
    )
    .copy()
)

print(
    f"Unique month-12 observation points: "
    f"{observation_df['loan_sequence_number'].nunique():,}"
)

Unique month-12 observation points: 45,127


#### Construct default_12m

In [13]:
# CONSTRUCT CALENDAR-BASED 12-MONTH OUTCOME WINDOW

# Convert YYYYMM reporting periods to monthly dates.
performance_history["reporting_date"] = pd.to_datetime(
    performance_history["monthly_reporting_period"]
        .astype("Int64")
        .astype(str),
    format="%Y%m",
    errors="coerce"
)

observation_df["observation_date"] = pd.to_datetime(
    observation_df["monthly_reporting_period"]
        .astype("Int64")
        .astype(str),
    format="%Y%m",
    errors="coerce"
)

# Exclude mortgages already seriously delinquent at observation.
observation_df = observation_df[
    observation_df["current_loan_delinquency_status"] < 3
].copy()

# Attach the unique observation date to each mortgage's history.
future_history = performance_history.merge(
    observation_df[
        ["loan_sequence_number", "observation_date"]
    ],
    on="loan_sequence_number",
    how="inner"
)

# Define the exact 12-calendar-month outcome window.
future_history["window_end"] = (
    future_history["observation_date"]
    + pd.DateOffset(months=12)
)

future_12m = future_history[
    (future_history["reporting_date"] > future_history["observation_date"]) &
    (future_history["reporting_date"] <= future_history["window_end"])
].copy()

# Serious delinquency event: 90+ DPD.
future_12m["default_event"] = (
    future_12m["current_loan_delinquency_status"] >= 3
).astype(int)

# Construct one target per mortgage.
default_target = (
    future_12m
    .groupby("loan_sequence_number")["default_event"]
    .max()
    .rename("default_12m")
    .reset_index()
)

# Create final month-12 performance dataset.
mortgage_performance = observation_df.merge(
    default_target,
    on="loan_sequence_number",
    how="inner"
)

print(f"Eligible mortgage records: {len(mortgage_performance):,}")
print(f"Unique loan IDs: {mortgage_performance['loan_sequence_number'].nunique():,}")

print("\nTarget distribution:")
print(mortgage_performance["default_12m"].value_counts())

print(
    f"\n12-month default rate: "
    f"{mortgage_performance['default_12m'].mean() * 100:.2f}%"
)

Eligible mortgage records: 44,098
Unique loan IDs: 44,098

Target distribution:
default_12m
0    42918
1     1180
Name: count, dtype: int64

12-month default rate: 2.68%


### Build the shared analytical dataset

##### Merge origination and month-12 information

In [14]:
# BUILD SHARED MORTGAGE ANALYTICAL DATASET
# Business objective:
# Combine:
#   1. Original borrower / underwriting characteristics
#   2. Original mortgage / property characteristics
#   3. Performance information available at month 12
#   4. Future 12-month default outcome

# Result: One analytical record per eligible mortgage.
mortgage_analytical = orig_df.merge(
    mortgage_performance,
    on="loan_sequence_number",
    how="inner",
    suffixes=("_orig", "_obs")
)

print(
    f"Final analytical dataset shape: "
    f"{mortgage_analytical.shape}"
)

display(mortgage_analytical.head())

Final analytical dataset shape: (44098, 43)


,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mi_percentage,number_of_units,occupancy_status,original_cltv,original_dti,...,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_maturity,current_interest_rate,current_deferred_upb,estimated_ltv,interest_bearing_upb,observation_date,default_12m
0,693,201803,N,203302,24340.0,0,1,P,80,41,...,124919.91,0.0,12,168,3.250,0.0,69,124919.91,2019-02-01,0
1,757,201803,Y,204802,NaN,25,1,P,97,10,...,26751.75,0.0,12,348,4.000,0.0,49,26751.75,2019-02-01,0
2,807,201803,N,203302,NaN,0,1,P,26,38,...,45632.04,0.0,12,168,3.250,0.0,22,45632.04,2019-02-01,0
3,812,201803,N,203302,19340.0,0,1,P,14,14,...,31653.56,0.0,12,168,3.750,0.0,12,31653.56,2019-02-01,0
4,661,201803,N,204802,41660.0,0,1,P,61,45,...,444946.15,0.0,12,348,3.875,0.0,62,444946.15,2019-02-01,0


##### **Important Insights about mortgage_analytical dataset**

One row represents one mortgage observed after 12 months of performance history. The record combines origination characteristics, information known at the 12-month observation point, and a binary outcome indicating whether the mortgage reaches 90+ days past due during the subsequent 12 months.

The analytical population consists of mortgages that have not yet defaulted after their first 12 months. Among these loans, 3.29% reach 90+ days past due during the subsequent 12 months.


##### **Business roles of our main variables**

| Business dimension       | Examples                                                                  | Purpose                                      |
| ------------------------ | ------------------------------------------------------------------------- | -------------------------------------------- |
| **Borrower credit risk** | `credit_score`, `original_dti`, `number_of_borrowers`                     | Financial capacity and creditworthiness      |
| **Loan leverage**        | `original_ltv`, `original_cltv`, `original_upb`                           | Initial exposure and borrower equity         |
| **Loan structure**       | `original_interest_rate`, `original_loan_term`, `loan_purpose`, `channel` | Characteristics of the credit agreement      |
| **Property**             | `property_type`, `occupancy_status`, `property_state`, `number_of_units`  | Characteristics of the collateral            |
| **Current exposure**     | `current_actual_upb`, `interest_bearing_upb`, `current_deferred_upb`      | Mortgage balance at observation              |
| **Current risk**         | `current_loan_delinquency_status`, `estimated_ltv`                        | Mortgage condition at observation            |
| **Time**                 | `loan_age`, `remaining_months_to_maturity`                                | Position in mortgage lifecycle               |
| **Identifier**           | `loan_sequence_number`                                                    | Record linkage only—not a predictive feature |
| **Target**               | `default_12m`                                                             | Default within the next 12 months            |


### Save the prepared PD dataset '*mortgage_pd_analytical.csv*' 

In [15]:
# SAVE MORTGAGE PD ANALYTICAL DATASET
# Business purpose:
# Persist the prepared loan-level dataset used for Mortgage PD model development.

# Each row represents an eligible mortgage observed at loan age 12, with default_12m representing whether the mortgage
# reaches 90+ DPD during the subsequent 12 months.

from pathlib import Path

# Define processed-data directory.
PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "freddie_mac"
    / "2018"
)

# Create directory if it does not already exist.
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Define output file.
PD_ANALYTICAL_PATH = (
    PROCESSED_DIR
    / "mortgage_pd_analytical.csv"
)

# Save without the pandas index because it has no business meaning.
mortgage_analytical.to_csv(
    PD_ANALYTICAL_PATH,
    index=False
)

print(
    f"Mortgage PD analytical dataset saved:\n"
    f"{PD_ANALYTICAL_PATH}"
)

print(
    f"\nDataset shape: "
    f"{mortgage_analytical.shape}"
)

Mortgage PD analytical dataset saved:
c:\Users\guima\OneDrive\Documents\AI_project\banking_services_ai_assistant\data\processed\freddie_mac\2018\mortgage_pd_analytical.csv

Dataset shape: (44098, 43)


### **EAD analytical dataset construction**

**Business logic**

Our target is:

$$ EAD = Current\ Actual\ UPB\ at\ first\ 90+\ DPD $$

But we cannot use information from the default month as predictors. We need a prediction point before default.

For EAD v1,the month immediately before the loan first reaches 90+ DPD will be using:

This answers a practical question:

Given the mortgage's characteristics immediately before serious delinquency/default, what exposure will the bank have when default occurs?

##### Extract first default event and prior-month observation

In [20]:
# BUILD EAD DEFAULT-EVENT DATASET

# Business objective:
# For each mortgage reaching 90+ DPD:
#   1. Identify the FIRST serious-delinquency event.
#   2. Capture Current Actual UPB at that event as EAD.
#   3. Capture the immediately preceding monthly observation as the prediction point.

# The raw performance file is processed in chunks to respect local memory constraints.
ead_columns = [
    "loan_sequence_number",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_maturity",
    "current_interest_rate",
    "current_deferred_upb",
    "estimated_ltv",
    "interest_bearing_upb"
]

# Store only records that could contribute to EAD construction.
ead_chunks = []

for chunk in pd.read_csv(
    PERF_PATH,
    sep="|",
    header=None,
    names=PERF_COLUMNS,
    usecols=ead_columns,
    chunksize=100_000,
    low_memory=False
):

    # Convert key modeling fields to numeric format.
    numeric_columns = [
        "current_actual_upb",
        "current_loan_delinquency_status",
        "loan_age",
        "remaining_months_to_maturity",
        "current_interest_rate",
        "current_deferred_upb",
        "estimated_ltv",
        "interest_bearing_upb"
    ]

    for column in numeric_columns:
        chunk[column] = pd.to_numeric(
            chunk[column],
            errors="coerce"
        )

    # Keep mortgages that belong to the population known to have reached our default definition.
    chunk = chunk[
        chunk["loan_sequence_number"].isin(default_loans)
    ].copy()

    ead_chunks.append(chunk)


# Combine performance histories only for defaulted mortgages.
ead_history = pd.concat(
    ead_chunks,
    ignore_index=True
)

# Ensure observations are in chronological order.
ead_history = ead_history.sort_values(
    ["loan_sequence_number", "loan_age"]
).reset_index(drop=True)

print(
    f"Performance records retained for EAD construction: "
    f"{len(ead_history):,}"
)

Performance records retained for EAD construction: 182,278


##### Construct EAD target and pre-default features

In [ ]:
# CONSTRUCT EAD TARGET AND PRE-DEFAULT OBSERVATION

# Business objective:
# For each mortgage, identify its FIRST 90+ DPD event and use the immediately preceding performance record as the
# prediction point.

# EAD target: Current Actual UPB at first 90+ DPD.

# Using shift() avoids duplicate matches that can occur when joining records using loan_age alone.

# 1. Sort each mortgage's history chronologically

ead_history = ead_history.sort_values(
    ["loan_sequence_number", "monthly_reporting_period"]
).reset_index(drop=True)

# 2. Create previous-month characteristics
# shift(1) retrieves the immediately preceding performance observation belonging to the same mortgage.

pre_default_features = [
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_maturity",
    "current_interest_rate",
    "current_deferred_upb",
    "estimated_ltv",
    "interest_bearing_upb"
]

for column in pre_default_features:

    ead_history[f"pre_default_{column}"] = (
        ead_history
        .groupby("loan_sequence_number")[column]
        .shift(1)
    )

# 3. Identify default observations
ead_history["default_event"] = (
    ead_history["current_loan_delinquency_status"] >= 3
)

# 4. Keep FIRST default event for each mortgage
first_default = (
    ead_history[
        ead_history["default_event"]
    ]
    .drop_duplicates(
        subset="loan_sequence_number",
        keep="first"
    )
    .copy()
)

# 5. Define EAD
# Outstanding principal balance at first 90+ DPD.

first_default["ead_at_default"] = (
    first_default["current_actual_upb"]
)

# 6. Keep loans having a valid preceding observation
ead_performance = first_default[
    first_default["pre_default_monthly_reporting_period"].notna()
].copy()


print(
    f"Unique first-default mortgages: "
    f"{first_default['loan_sequence_number'].nunique():,}"
)

print(
    f"EAD records with pre-default observation: "
    f"{ead_performance['loan_sequence_number'].nunique():,}"
)

print("\nEAD target summary:")
display(
    ead_performance["ead_at_default"].describe()
)

Unique first-default mortgages: 2,708
EAD records with pre-default observation: 2,708

EAD target summary:


count      2708.000000
mean     222870.133061
std      126114.711424
min           0.000000
25%      127554.720000
50%      194971.565000
75%      299278.430000
max      707027.470000
Name: ead_at_default, dtype: float64

##### Add origination characteristics

In [24]:
# BUILD FINAL EAD ANALYTICAL DATASET
# Combine origination characteristics with information known immediately before first default.

mortgage_ead_analytical = orig_df.merge(
    ead_performance,
    on="loan_sequence_number",
    how="inner",
    suffixes=("_orig", "_pre_default")
)

print(
    f"Final EAD analytical dataset shape: "
    f"{mortgage_ead_analytical.shape}"
)

display(
    mortgage_ead_analytical.head()
)

Final EAD analytical dataset shape: (2708, 52)


,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mi_percentage,number_of_units,occupancy_status,original_cltv,original_dti,...,pre_default_monthly_reporting_period,pre_default_current_actual_upb,pre_default_current_loan_delinquency_status,pre_default_loan_age,pre_default_remaining_months_to_maturity,pre_default_current_interest_rate,pre_default_current_deferred_upb,pre_default_estimated_ltv,pre_default_interest_bearing_upb,ead_at_default
0,705,201803,Y,204802,17820.0,30,1,P,95,9,...,202012.0,265071.57,2.0,34.0,326.0,3.875,0.0,67.0,265071.57,265071.57
1,642,201803,N,204802,13900.0,12,1,P,85,43,...,202006.0,307531.09,2.0,28.0,332.0,4.250,0.0,79.0,307531.09,307531.09
2,686,201803,Y,204802,38900.0,30,1,P,95,45,...,202006.0,408549.90,2.0,28.0,332.0,4.375,0.0,77.0,408549.90,408549.90
3,782,201805,N,204804,31084.0,30,1,P,95,41,...,202006.0,343714.77,2.0,26.0,334.0,4.125,0.0,88.0,343714.77,343714.77
4,781,201804,N,203803,37900.0,0,1,P,80,47,...,202005.0,127493.82,2.0,26.0,214.0,4.000,0.0,78.0,127493.82,127493.82


##### Save the EAD analytical dataset

In [25]:
# SAVE MORTGAGE EAD ANALYTICAL DATASET

# Business purpose:
# Persist the prepared dataset used for EAD model development.

# Each row represents a mortgage that reaches first 90+ DPD.
# Predictors include:
#   - Original underwriting characteristics
#   - Mortgage characteristics immediately before default

# Target: ead_at_default = Current Actual UPB at first 90+ DPD

EAD_ANALYTICAL_PATH = (
    PROCESSED_DIR
    / "mortgage_ead_analytical.csv"
)

# Save the prepared EAD dataset.
mortgage_ead_analytical.to_csv(
    EAD_ANALYTICAL_PATH,
    index=False
)

print(
    f"Mortgage EAD analytical dataset saved:\n"
    f"{EAD_ANALYTICAL_PATH}"
)

print(
    f"\nDataset shape: "
    f"{mortgage_ead_analytical.shape}"
)

Mortgage EAD analytical dataset saved:
c:\Users\guima\OneDrive\Documents\AI_project\banking_services_ai_assistant\data\processed\freddie_mac\2018\mortgage_ead_analytical.csv

Dataset shape: (2708, 52)
